In [211]:
import numpy as np
import pandas as pd

In [226]:
# データを読み込みます
df = pd.read_csv('../Life.csv',parse_dates=['Timestamp'])
apps = 'fuser_life'

In [227]:
def fuser_Cycle(df):
    # 並び替え
    df = df.sort_values(['Deviceid', 'Timestamp']).reset_index(drop = True)
    # 列追加
    df['Unit'] = 'Fuser'
    df['Deviceid_Shift'] = df.Deviceid.shift()
    df['Life_Diff'] = df['Life'].diff()
    df['Page_Diff'] = df['Total Page Count'].diff()
    df['Rol_Diff'] = df['Total Rol Distance'].diff()
    # Deviceidの切替りのデータをnanに置き換える
    df.loc[df.Deviceid != df.Deviceid_Shift, 'Life_Diff'] = np.nan
    df.loc[df.Deviceid != df.Deviceid_Shift, 'Page_Diff'] = np.nan
    df.loc[df.Deviceid != df.Deviceid_Shift, 'Rol_Diff'] = np.nan
    # 交換タイミングを計算
    df.loc[(df.Life_Diff > 0) & (df.Page_Diff < 0) & (df.Rol_Diff < 0), 'Event'] = '交換'
    # 不要な列を削除
    df = df.drop(['Deviceid_Shift', 'Life_Diff', 'Page_Diff', 'Rol_Diff'], axis=1)
    return df

In [228]:
def life_cycle(df):
    for row in range(len(df)):
        if row == 0:
            cycle = 1
            count_new = 0
            df.loc[row,'Cycle'] = cycle
            df.loc[row,'Reset_Count'] = df.loc[row,'Pagecount']
        else:
            # 同じ本体、新品以外
            if df.loc[row,'Event'] != '交換' and df.loc[row,'Deviceid'] == df.loc[row-1,'Deviceid']:
                df.loc[row,'Cycle'] = cycle
                df.loc[row,'Reset_Count'] = df.loc[row,'Pagecount'] - count_new
            # 同じ本体、新品
            elif df.loc[row,'Event'] == '交換' and df.loc[row,'Deviceid'] == df.loc[row-1,'Deviceid']:
                df.loc[row,'Cycle'] = cycle
                df.loc[row,'Reset_Count'] = df.loc[row,'Pagecount'] - count_new
                # 2サイクル目以降で交換後1000枚以内の再交換はnanに置き換える
                if cycle == 1:
                    cycle += 1
                    count_new = df.loc[row,'Pagecount']
                elif df.loc[row,'Pagecount'] - count_new > 1000 :
                    cycle += 1
                    count_new = df.loc[row,'Pagecount']
                else:
                    df.loc[row,'Event'] = np.nan
            # 違う本体への切替り、新品以外
            elif df.loc[row,'Event'] != '交換' and df.loc[row,'Deviceid'] != df.loc[row-1,'Deviceid']:
                cycle = 1
                count_new = 0
                df.loc[row,'Cycle'] = cycle
                df.loc[row,'Reset_Count'] = df.loc[row,'Pagecount'] - count_new
            # 違う本体への切替り、新品
            elif df.loc[row,'Event'] == '交換' and df.loc[row,'Deviceid'] != df.loc[row-1,'Deviceid']:
                cycle = 1
                count_new = 0
                df.loc[row,'Cycle'] = cycle
                df.loc[row,'Reset_Count'] = df.loc[row,'Pagecount'] - count_new
                # 交換後1000枚以内の再交換はnanに置き換える
                if cycle == 1:
                    cycle += 1
                    count_new = df.loc[row,'Pagecount']
                elif df.loc[row,'Pagecount'] - count_new > 1000 :
                    cycle += 1
                    count_new = df.loc[row,'Pagecount']
                else:
                    df.loc[row,'Event'] = np.nan
    return df

In [229]:
# 交換タイミングの取得
if apps == 'fuser_life':
    df = fuser_Cycle(df) 
elif apps == 'itb_life':
    df = fuser_Cycle(df) 

# 交換サイクルとリセットページカウントの算出
df = life_cycle(df)
df

,Deviceid,Timestamp,Pagecount,Life,Total Page Count,Total Rol Distance,Unit,Event,Cycle,Reset_Count
0,aaa,2020-01-01,4000,74,4000,4000,Fuser,NaN,1.0,4000.0
1,aaa,2020-01-02,8000,1,8000,8000,Fuser,NaN,1.0,8000.0
2,aaa,2020-01-03,11000,-5,12000,12000,Fuser,NaN,1.0,11000.0
3,aaa,2020-01-04,13000,100,0,0,Fuser,交換,1.0,13000.0
4,aaa,2020-01-05,13000,100,0,0,Fuser,NaN,2.0,0.0
5,aaa,2020-01-06,15000,72,4000,4000,Fuser,NaN,2.0,2000.0
6,bbb,2020-01-07,7,100,2,0,Fuser,NaN,1.0,7.0
7,bbb,2020-01-08,8,100,0,40000,Fuser,NaN,1.0,8.0
8,ccc,2020-01-16,4,100,4,4,Fuser,NaN,1.0,4.0
9,ccc,2020-01-17,5000,-8,4500,4500,Fuser,NaN,1.0,5000.0


In [234]:
df

,Deviceid,Timestamp,Pagecount,Life,Total Page Count,Total Rol Distance,Unit,Event,Cycle,Reset_Count,Life2
0,aaa,2020-01-01,4000,74,4000,4000,Fuser,NaN,1.0,4000.0,NaN
1,aaa,2020-01-02,8000,1,8000,8000,Fuser,NaN,1.0,8000.0,1.0
2,aaa,2020-01-03,11000,-5,12000,12000,Fuser,NaN,1.0,11000.0,-5.0
3,aaa,2020-01-04,13000,100,0,0,Fuser,交換,1.0,13000.0,NaN
4,aaa,2020-01-05,13000,100,0,0,Fuser,NaN,2.0,0.0,NaN
5,aaa,2020-01-06,15000,72,4000,4000,Fuser,NaN,2.0,2000.0,NaN
6,bbb,2020-01-07,7,100,2,0,Fuser,NaN,1.0,7.0,NaN
7,bbb,2020-01-08,8,100,0,40000,Fuser,NaN,1.0,8.0,NaN
8,ccc,2020-01-16,4,100,4,4,Fuser,NaN,1.0,4.0,NaN
9,ccc,2020-01-17,5000,-8,4500,4500,Fuser,NaN,1.0,5000.0,-8.0


In [230]:
df['Life2'] = df.Life.apply(lambda x : x if x<5 else np.nan)

In [233]:
df['Life3'] = df.groupby(['Deviceid','Cycle'])['Life2'].idxmax()

TypeError: incompatible index of inserted column with frame index

In [ ]:
# df['Life_Low'] = df.Pagecount.apply(lambda x: 'Life_Low' if x<=1 else np.nan)

In [66]:
# 各デバイス毎の一番最初と最後のデータのみの日付データを作成（エラー発生なし本体の計算と、各本体のデータ取得期間確認用）
# df_serial1 = pd.DataFrame()
# df_serial2 = pd.DataFrame()
# df_serial1['start'] = df.groupby('serial').start.min()
# df_serial2['start'] = df.groupby('serial').start.max()
# df_serial = pd.concat([df_serial1,df_serial2])
# df_serial = df_serial.reset_index()
# df_serial = df_serial.sort_values(['serial','start'])
# df_serial


In [67]:
# 各デバイス毎の一番最初と最後のデータのみを抽出（エラー発生なし本体の計算と、各本体のデータ取得期間確認用）
df_group = df.groupby('serial')
df_serial1 = df.loc[df_group['start'].idxmin(),:]
df_serial2 = df.loc[df_group['start'].idxmax(),:]
df_serial = pd.concat([df_serial1,df_serial2])
df_serial = df_serial.sort_values(['serial','start'])
df_serial = df_serial.reset_index(drop=True)
df_serial

,start,end,nr,serial,error_code,factor,print_count
0,2020-01-01,2020-01-02,500-1,aaa,1.0,NaN,1
1,2020-01-06,2020-03-20,600-1,aaa,2.0,NaN,6
2,2020-01-07,2020-01-07,701-1,bbb,4.0,NaN,7
3,2020-01-15,2020-01-07,801-1,bbb,4.0,JAM,15
4,2020-01-16,2020-01-07,801-1,ccc,8.0,JAM,16
5,2020-01-17,2020-01-07,801-1,ccc,1.0,JAM,17
6,2020-01-18,2020-02-07,600-1,eee,NaN,NaN,18
7,2020-01-24,2020-01-07,801-1,eee,NaN,JAM,24


In [68]:
# エラー発生時のインデックスリストを作成
index_list = df[~df.factor.isnull()].index.to_list()
index_list

[3, 4, 9, 10, 11, 14, 15, 16, 22, 23]

In [69]:
# エラー発生前後50行のインデックスリストを作成
index_max = len(df)
hold_list = []
for i in index_list:
    for j in range(-50,51):
        if i+j>=0 and i+j<index_max:
            hold_list.append(i+j)
hold_list = list(set(hold_list))
hold_list

[0,
 1,
 2,
 3,
 4,
 5,
 6,
 7,
 8,
 9,
 10,
 11,
 12,
 13,
 14,
 15,
 16,
 17,
 18,
 19,
 20,
 21,
 22,
 23]

In [70]:
# エラー発生前後50行のデータのみを残す
df.loc[df.index[hold_list],'Hold'] = 1
df = df.dropna(subset=['Hold'])
df = df.drop('Hold',axis=1)
df

,start,end,nr,serial,error_code,factor,print_count
0,2020-01-01,2020-01-02,500-1,aaa,1.0,NaN,1
1,2020-01-02,2020-01-07,500-2,aaa,2.0,NaN,2
2,2020-01-03,2020-02-07,500-3,aaa,4.0,NaN,3
3,2020-01-04,2020-01-10,600-1,aaa,8.0,PQ,4
4,2020-01-05,2020-02-20,600-1,aaa,3.0,Noise,5
5,2020-01-06,2020-03-20,600-1,aaa,2.0,NaN,6
6,2020-01-07,2020-01-07,701-1,bbb,4.0,NaN,7
7,2020-01-08,2020-01-02,500-1,bbb,8.0,NaN,8
8,2020-01-09,2020-01-07,500-2,bbb,1.0,NaN,9
9,2020-01-10,2020-02-07,500-3,bbb,2.0,JAM,10


In [73]:
df = pd.concat([df,df_serial])
df = df.sort_values(['serial','start'])
df = df.reset_index(drop=True)
df

,start,end,nr,serial,error_code,factor,print_count
0,2020-01-01,2020-01-02,500-1,aaa,1.0,NaN,1
1,2020-01-01,2020-01-02,500-1,aaa,1.0,NaN,1
2,2020-01-01,2020-01-02,500-1,aaa,1.0,NaN,1
3,2020-01-01,2020-01-02,500-1,aaa,1.0,NaN,1
4,2020-01-02,2020-01-07,500-2,aaa,2.0,NaN,2
5,2020-01-03,2020-02-07,500-3,aaa,4.0,NaN,3
6,2020-01-04,2020-01-10,600-1,aaa,8.0,PQ,4
7,2020-01-05,2020-02-20,600-1,aaa,3.0,Noise,5
8,2020-01-06,2020-03-20,600-1,aaa,2.0,NaN,6
9,2020-01-06,2020-03-20,600-1,aaa,2.0,NaN,6
